In [0]:
CATALOG = dbutils.widgets.get("catalog")

BRONZE = f"{CATALOG}.skybound_bronze.weather_reports_bronze"
SILVER = f"{CATALOG}.skybound_silver.weather_reports_silver"

In [0]:
# Get the current Version of the bronze Table
current_version = spark.sql(f"SELECT version FROM (DESCRIBE HISTORY {BRONZE} LIMIT 1)").collect()[0][0]

df_bronze_changed_data = spark.sql(f"SELECT * FROM table_changes('{BRONZE}', {current_version})")
# display(df_bronze_changed_data)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

# ── Transform ──────────────────────────────────────────────────────────────
transformed = (
    df_bronze_changed_data
    .withColumn("obs_time",      F.to_timestamp(F.col("obsTime").cast("long")))
    .withColumn("receipt_time",  F.to_timestamp("receiptTime"))
    .withColumn("temp_c",        F.col("temp").cast("double"))
    .withColumn("dewpoint_c",    F.col("dewp").cast("double"))
    .withColumn("wind_dir_degrees",
        F.when(F.col("wdir") == "VRB", F.lit(None))
         .otherwise(F.col("wdir").cast("int")))
    .withColumn("wind_speed_kt", F.col("wspd").cast("int"))
    .withColumn("wind_gust_kt",  F.col("wgst").cast("int"))
    .withColumn("visibility_sm",
        F.when(F.col("visib") == "6+", F.lit(6.0))
         .otherwise(F.col("visib").cast("double")))
    .withColumn("altim_hpa",     F.col("altim").cast("double"))
    .withColumn("cloud_base_ft", F.col("cloud_base_ft").try_cast("int"))
    .withColumn("latitude",      F.col("lat").cast("double"))
    .withColumn("longitude",     F.col("lon").cast("double"))
    .withColumn("elev_m",        F.col("elev").cast("int"))

    # Deduplicate: same station + same observation = same METAR
    .dropDuplicates(["icaoId", "obsTime"])
    .select(
        F.col("icaoId")              .alias("station_id"),
        F.col("name")                .alias("station_name"),
        "obs_time", "receipt_time",
        "temp_c", "dewpoint_c",
        "wind_dir_degrees", "wind_speed_kt", "wind_gust_kt",
        "visibility_sm", "altim_hpa",
        F.col("wxString")            .alias("wx_string"),
        "cover", "sky_cover", "cloud_base_ft",
        F.col("fltCat")              .alias("flight_category"),
        F.col("metarType")           .alias("metar_type"),
        "latitude", "longitude", "elev_m",
        F.col("rawOb")               .alias("raw_ob"),
        "_ingestion_timestamp"
    )
)

# ── MERGE into silver  ───────────────────
transformed.createOrReplaceTempView("weather_staging")

spark.sql(f"""
  MERGE INTO {SILVER} AS target
  USING weather_staging AS source
  ON target.station_id = source.station_id
  AND target.obs_time = source.obs_time

  WHEN NOT MATCHED THEN INSERT *
""")
